# Notebook 4E-bis : Test de Parallélisme avec Blueprint Complet

**Objectif** : Vérifier que le système complet (Blueprint → Controller → DaskBackend) parallélise correctement.

## Différence avec 04e

- **04e** : Teste directement le DaskBackend en créant manuellement des ComputeNodes
- **04e-bis** : Teste le système complet tel qu'utilisé en production (05a)

## Architecture du Test

- Crée **12 groupes fonctionnels indépendants** (comme dans un vrai modèle multi-espèces)
- Chaque groupe a **1 tâche sleep** qui dort pendant 0.1s
- Chaque groupe a sa propre variable d'état "tracer"
- Cette architecture simule un modèle réaliste avec plusieurs groupes fonctionnels parallèles

## Principe

- Déclarer le modèle via Blueprint (comme dans 05a)
- Exécuter via SimulationController avec backend Dask
- Comparer les performances avec différents nombres de workers

In [ ]:
import time
from datetime import datetime, timedelta
from pathlib import Path

import dask
import matplotlib.pyplot as plt
import numpy as np
import pint
import xarray as xr
from IPython.display import Markdown

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController

print("✅ Imports réussis")

## Configuration

In [ ]:
CONFIG = {
    "n_tasks": 12,
    "sleep_duration": 0.1,
    "grid_size": (100, 100),
    "n_timesteps": 1,
    "n_runs": 3,
    "workers_list": np.arange(1, 13),
}

EXPECTED_SEQ_TIME = CONFIG["n_tasks"] * CONFIG["sleep_duration"]
print(f"Temps séquentiel attendu : {EXPECTED_SEQ_TIME:.2f}s")
print(f"Temps parallèle attendu (12 workers) : {CONFIG['sleep_duration']:.2f}s")
print(f"Speedup idéal max : {CONFIG['n_tasks']}×")

## Création des Fonctions Sleep

In [ ]:
def make_sleep_function(task_id: int, sleep_duration: float):
    def compute_sleep_tendency(state: xr.DataArray) -> dict[str, xr.DataArray]:
        time.sleep(sleep_duration)
        tendency = xr.zeros_like(state)
        return {"tendency": tendency}

    compute_sleep_tendency.__name__ = f"sleep_task_{task_id}"
    return compute_sleep_tendency


sleep_functions = [
    make_sleep_function(i, CONFIG["sleep_duration"]) for i in range(CONFIG["n_tasks"])
]

print(f"✅ {len(sleep_functions)} fonctions sleep créées")

## Configuration du Blueprint

In [ ]:
def configure_sleep_model(bp: Blueprint):
    # Créer un groupe fonctionnel par tâche (comme plusieurs groupes zooplankton/fish/etc.)
    for i, func in enumerate(sleep_functions):
        bp.register_group(
            group_prefix=f"FunctionalGroup_{i}",
            units=[
                {
                    "func": func,
                    "input_mapping": {"state": "tracer"},
                    "output_mapping": {"tendency": "tracer_tendency"},
                    "output_tendencies": {"tendency": "tracer"},
                    "output_units": {"tendency": "dimensionless"},
                }
            ],
            state_variables={
                "tracer": {
                    "dims": ("y", "x"),
                    "units": "dimensionless",
                }
            },
            parameters={},
        )


bp_test = Blueprint()
configure_sleep_model(bp_test)

print(f"✅ Blueprint configuré avec {CONFIG['n_tasks']} groupes fonctionnels")
print(f"   Chaque groupe a 1 tâche sleep")
print("\nDiagramme Mermaid du Blueprint:")
Markdown("```mermaid\n" + bp_test.export_mermaid() + "\n```")

## Préparation des Forcings et État Initial

In [ ]:
import pandas as pd

ny, nx = CONFIG["grid_size"]
n_timesteps = CONFIG["n_timesteps"]

# Créer des vrais timestamps pour le temps
time_range = pd.date_range("2000-01-01", periods=n_timesteps, freq="1s")

forcings = xr.Dataset(
    coords={
        "time": time_range,
        "y": np.arange(ny),
        "x": np.arange(nx),
    }
)
forcings["dt"] = CONFIG["sleep_duration"]

# Créer l'état initial pour chaque groupe fonctionnel
# Chaque groupe a sa propre variable "tracer"
initial_states = {}
for i in range(CONFIG["n_tasks"]):
    initial_states[f"FunctionalGroup_{i}"] = xr.Dataset(
        {
            "tracer": xr.DataArray(
                np.ones((ny, nx)), dims=["y", "x"], attrs={"units": "dimensionless"}
            )
        }
    )

print(f"États initiaux créés pour {len(initial_states)} groupes fonctionnels")
print(f"Forcings créés : {list(forcings.coords)}")

## Configuration de la Simulation

In [ ]:
sim_config = SimulationConfig(
    start_date="2000-01-01",
    end_date="2000-01-01T00:00:01",
    timestep=timedelta(seconds=1),
)

print(f"Configuration : 1 timestep")

## Test 1 : Exécution Séquentielle (Baseline)

In [ ]:
def run_simulation_sequential(n_runs=3):
    times = []

    # Préparer les output_variables pour tous les groupes
    output_variables = {f"FunctionalGroup_{i}": ["tracer"] for i in range(CONFIG["n_tasks"])}

    for _ in range(n_runs):
        controller = SimulationController(sim_config, backend="sequential")
        controller.setup(
            configure_sleep_model,
            forcings,
            initial_state=initial_states,
            parameters={f"FunctionalGroup_{i}": {} for i in range(CONFIG["n_tasks"])},
            output_variables=output_variables,
        )

        t_start = time.perf_counter()
        controller.run()
        t_end = time.perf_counter()
        times.append(t_end - t_start)

    return np.mean(times), np.std(times)


print("Exécution séquentielle...")
t_seq_mean, t_seq_std = run_simulation_sequential(CONFIG["n_runs"])
print(f"Temps séquentiel : {t_seq_mean:.3f}s ± {t_seq_std:.3f}s")
print(f"Temps attendu    : {EXPECTED_SEQ_TIME:.3f}s")
print(f"Écart            : {abs(t_seq_mean - EXPECTED_SEQ_TIME) / EXPECTED_SEQ_TIME * 100:.1f}%")

## Test 2 : Exécution Parallèle avec Backend Dask

In [ ]:
def run_simulation_dask(num_workers, n_runs=3):
    times = []

    # Préparer les output_variables pour tous les groupes
    output_variables = {f"FunctionalGroup_{i}": ["tracer"] for i in range(CONFIG["n_tasks"])}

    for _ in range(n_runs):
        with dask.config.set(scheduler="threads", num_workers=num_workers):
            controller = SimulationController(sim_config, backend="dask")
            controller.setup(
                configure_sleep_model,
                forcings,
                initial_state=initial_states,
                parameters={f"FunctionalGroup_{i}": {} for i in range(CONFIG["n_tasks"])},
                output_variables=output_variables,
            )

            t_start = time.perf_counter()
            controller.run()
            t_end = time.perf_counter()
            times.append(t_end - t_start)

    return np.mean(times), np.std(times)


results = []

print("Test de parallélisme avec Blueprint + Controller + DaskBackend")
print("=" * 60)
print(f"{'Workers':<10} {'Temps (s)':<15} {'Speedup':<10} {'Efficacité':<10}")
print("-" * 60)

for n_workers in CONFIG["workers_list"]:
    print(f"Testing {n_workers} workers...", end=" ")
    t_mean, t_std = run_simulation_dask(n_workers, CONFIG["n_runs"])
    speedup = t_seq_mean / t_mean
    efficiency = speedup / n_workers * 100

    results.append(
        {
            "workers": n_workers,
            "time_mean": t_mean,
            "time_std": t_std,
            "speedup": speedup,
            "efficiency": efficiency,
        }
    )

    print(f"{n_workers:<10} {t_mean:.3f} ± {t_std:.3f}   {speedup:.2f}×       {efficiency:.1f}%")

print("=" * 60)

## Visualisation des Résultats

In [ ]:
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 11,
        "legend.fontsize": 9,
        "lines.linewidth": 1.5,
        "lines.markersize": 6,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

workers = [r["workers"] for r in results]
speedups = [r["speedup"] for r in results]
efficiencies = [r["efficiency"] for r in results]

ax1 = axes[0]
ax1.plot(workers, workers, "k--", label="Idéal (y=x)", alpha=0.7)
ax1.plot(workers, speedups, "o-", color="#0077BB", label="Mesuré (Blueprint)")
ax1.set_xlabel("Nombre de Workers")
ax1.set_ylabel("Speedup")
ax1.set_title("Strong Scaling (Blueprint + Controller + Dask)")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, max(workers) + 1)
ax1.set_ylim(0, max(workers) + 1)

ax2 = axes[1]
ax2.axhline(100, color="k", linestyle="--", alpha=0.7, label="Idéal (100%)")
ax2.plot(workers, efficiencies, "s-", color="#EE7733", label="Mesuré (Blueprint)")
ax2.set_xlabel("Nombre de Workers")
ax2.set_ylabel("Efficacité (%)")
ax2.set_title("Efficacité Parallèle")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, max(workers) + 1)
ax2.set_ylim(0, 120)

plt.tight_layout()
plt.show()

output_dir = Path("figures")
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / "fig_04e_sleep_parallelism_blueprint.png", dpi=300, bbox_inches="tight")
fig.savefig(output_dir / "fig_04e_sleep_parallelism_blueprint.pdf", bbox_inches="tight")
print("✅ Figures sauvegardées")

## Analyse des Résultats

In [ ]:
max_speedup = max(speedups)
max_workers = results[speedups.index(max_speedup)]["workers"]
avg_efficiency = np.mean(efficiencies)

print("=" * 70)
print("ANALYSE DES RÉSULTATS (SYSTÈME COMPLET)")
print("=" * 70)

print(f"\nSpeedup maximum   : {max_speedup:.2f}× avec {max_workers} workers")
print(f"Efficacité moyenne: {avg_efficiency:.1f}%")

print("\n--- DIAGNOSTIC ---")

if max_speedup >= CONFIG["n_tasks"] * 0.8:
    print("✅ EXCELLENT : Le speedup est proche de l'idéal.")
    print("   Le système complet fonctionne parfaitement.")
    conclusion = "SYSTEM_OK"
elif max_speedup >= CONFIG["n_tasks"] * 0.5:
    print("⚠️  MODÉRÉ : Le speedup est correct mais pas optimal.")
    print("   Il y a probablement un overhead dans Blueprint/Controller.")
    conclusion = "PARTIAL"
else:
    print("❌ MAUVAIS : Le speedup est très faible.")
    print("   Le système complet a un problème.")
    conclusion = "SYSTEM_ISSUE"

t_ideal_parallel = CONFIG["sleep_duration"]
t_best_measured = min(
    [r["time_mean"] for r in results if r["workers"] >= CONFIG["n_tasks"]],
    default=results[-1]["time_mean"],
)
overhead = t_best_measured - t_ideal_parallel

print(f"\n--- OVERHEAD SYSTÈME COMPLET ---")
print(f"Temps idéal (12 workers) : {t_ideal_parallel:.3f}s")
print(f"Temps mesuré             : {t_best_measured:.3f}s")
print(f"Overhead estimé          : {overhead:.3f}s ({overhead / t_best_measured * 100:.1f}%)")

## Génération du Résumé

In [ ]:
summary_lines = [
    "=" * 100,
    "NOTEBOOK 4E-BIS: TEST DE PARALLÉLISME AVEC BLUEPRINT COMPLET",
    "=" * 100,
    "",
    f"DATE: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "",
    "OBJECTIF:",
    "-" * 100,
    "Vérifier que le système complet (Blueprint → SimulationController → DaskBackend)",
    "parallélise correctement, en utilisant la même architecture que les simulations réelles.",
    "",
    "ARCHITECTURE:",
    "-" * 100,
    f"Nombre de groupes fonctionnels : {CONFIG['n_tasks']} (indépendants)",
    "Tâches par groupe              : 1 (fonction sleep)",
    "Cette architecture simule un modèle multi-espèces réaliste.",
    "",
    "CONFIGURATION:",
    "-" * 100,
    f"Durée sleep/tâche     : {CONFIG['sleep_duration']}s",
    f"Grille                : {CONFIG['grid_size'][0]}×{CONFIG['grid_size'][1]}",
    f"Workers testés        : {CONFIG['workers_list']}",
    "",
    "RÉSULTATS:",
    "-" * 100,
    f"{'Workers':<10} {'Temps (s)':<15} {'Speedup':<12} {'Efficacité (%)':<15}",
    "-" * 60,
]

for r in results:
    summary_lines.append(
        f"{r['workers']:<10} {r['time_mean']:<15.3f} {r['speedup']:<12.2f} {r['efficiency']:<15.1f}"
    )

summary_lines.extend(
    [
        "",
        "ANALYSE:",
        "-" * 100,
        f"Speedup maximum       : {max_speedup:.2f}× avec {max_workers} workers",
        f"Efficacité moyenne    : {avg_efficiency:.1f}%",
        f"Overhead système      : {overhead:.3f}s",
        "",
        "CONCLUSION:",
        "-" * 100,
    ]
)

if conclusion == "SYSTEM_OK":
    summary_lines.extend(
        [
            "✅ Le système complet fonctionne parfaitement.",
            "   Toute la chaîne Blueprint → Controller → DaskBackend parallélise correctement.",
            "   La parallélisation entre groupes fonctionnels est efficace.",
        ]
    )
elif conclusion == "PARTIAL":
    summary_lines.append("⚠️  Le système fonctionne mais avec un overhead.")
else:
    summary_lines.append("❌ Le système complet a un problème de parallélisation.")

summary_lines.extend(
    [
        "",
        "=" * 100,
    ]
)

summary = "\n".join(summary_lines)

with open(output_dir / "notebook_04e_blueprint_summary.txt", "w") as f:
    f.write(summary)

print(summary)